**University of Antananarivo**

## Cryptography Project by RANDRIAMAMONJY Tokiniaina

**Introduction:**

This project aims to demonstrate the functioning of the RSA and ElGamal protocols in cryptography.  
It was made possible thanks to two fundamental foundations: computer science (using Python) and mathematics (based on group theory).

We are going to work on an AIMS exercise:<br>

![exercise 2](fig1.png)

*baby-step*:

In [55]:
import math  #x = v*m + u
def baby_step(g,N,h,mod): #solve g^x = h (mod)
    m=math.ceil(math.sqrt(N))
    baby={}
    for u in range(m):
        baby[pow(g,u,mod)]=u
        
    value=h
    g_1=pow(g,-m,mod)
    for v in range(m):
        if value in baby:
           u=baby[value]
           return u+m*v
        value=(value*g_1)%mod

    raise ValueError("Logarithme discret non trouvé")
        

*Chinese Remainder Theorem:*

In [56]:
def crt(congruences):
    N=1
    for _,ni in congruences:
        N*=ni
    x=0
    for ai,ni in congruences:
        Ni=N//ni
        yi=pow(Ni,-1,ni)
        ci=Ni*yi
        x+=ai*ci
    return x % N

*Factorization of n:*

In [57]:
def factors(n):
    factors = []
    d = 2
    while d*d <= n:
        s = 0
        while n % d == 0:
            s += 1
            n //= d
        if s > 0:
            factors.append((d, s))
        d += 1
    if n > 1:
        factors.append((n, 1))
    return factors

*Silver_polying_hellman:*

In [58]:
import math
def pohlig_hellman(g, h, n):
    prime_factors = factors(n)
    congruences = []

    for p, e in prime_factors:
        pe = p**e
        g_i = pow(g, n//pe, n+1)  
        h_i = pow(h, n//pe, n+1)
        x_i = baby_step(g_i, pe, h_i, n+1)
        congruences.append((x_i, pe))
    x = crt(congruences)
    return x

In [59]:
import math
def pohlig_hellman(g, h, n):
    prime_factors = factors(n)
    congruences = []

    for p, e in prime_factors:
        pe = p**e
        g_i = pow(g, n//pe, n+1)  
        h_i = pow(h, n//pe, n+1)
        x_i = baby_step(g_i, pe, h_i, n+1)
        congruences.append((x_i, pe))
    x = crt(congruences)
    return x

*Determination of Bob's key:*

In [60]:
p = 123456789987654353003
g = 123456789      
h = 39318628345168608817
n = p - 1  

x = pohlig_hellman(g, h, n)
print(f"g^{x} ≡ h (mod {p}) ? -> {pow(g, x, p) == h}")
print("Solution x =", x)

g^5191 ≡ h (mod 123456789987654353003) ? -> True
Solution x = 5191


In [61]:
p = 123456789987654353003
g = 123456789
b = 5191 

ciphertexts = [
    (83025882561049910713, 66740266984208729661),
    (117087132399404660932, 44242256035307267278),
    (67508282043396028407, 77559274822593376192),
    (60938739831689454113, 14528504156719159785),
    (5059840044561914427, 59498668430421643612),
    (92232942954165956522, 105988641027327945219),
    (97102226574752360229, 46166643538418294423)
]

code_to_char = {
    11: 'A', 12: 'B', 13: 'C', 14: 'D', 15: 'E', 16: 'F', 17: 'G', 18: 'H', 19: 'I', 20: 'J',
    21: 'K', 22: 'L', 23: 'M', 24: 'N', 25: 'O', 26: 'P', 27: 'Q', 28: 'R', 29: 'S', 30: 'T',
    31: 'U', 32: 'V', 33: 'W', 34: 'X', 35: 'Y', 36: 'Z',
    41: ' ', 42: '0', 43: '.', 44: ',', 45: '?'
}

def decrypt(c1, c2, b, p):
    c1_b = pow(c1, b, p)
    inv = pow(c1_b, -1, p)
    m = (c2 * inv) % p
    return m

def decode_message(m):
    m_str = str(m).zfill(20) 
    chars = []
    for i in range(0, 20, 2):
        code = int(m_str[i:i+2])
        if code in code_to_char:
            chars.append(code_to_char[code])
        else:
            chars.append('?')  
    return ''.join(chars)
message_parts = []
for c1, c2 in ciphertexts:
    m = decrypt(c1, c2, b, p)
    part = decode_message(m)
    message_parts.append(part)

# Full message
full_message = ''.join(message_parts)
print("decryted message: ",full_message)

decryted message:  IN GALOIS FIELDS, FULL OF FLOWERS, PRIMITIVE ELEMENTS DANCE FOR HOURS.


**Here we successfully decrypted the requested message (using the ElGamal protocol).**

**Now let's take a look at the RSA protocol:**

The RSA protocol is mainly based on two keys, namely the prime numbers p and q.  
We will try to generate p and q that are robust against attacks such as:  
- GNFS  
- Wiener  
- Fermat  
- Pollard p-1  
...

## Sophie germain prime algorithm

In [62]:
import random
from sympy import isprime, mod_inverse
from math import isqrt

def generate_sophie_germain_prime(bit_length):
    while True:
        ell = random.getrandbits(bit_length - 1) | 1
        if isprime(ell):
            p = 2 * ell + 1
            if p.bit_length() == bit_length and isprime(p):
                return p, ell

def is_mersenne(n):
    k = 1
    while (2 ** k - 1) <= n:
        if 2 ** k - 1 == n:
            return True
        k += 1
    return False

p_bits = 1022
q_bits = 1026
p, ell_p = generate_sophie_germain_prime(p_bits)

N = 0
while N.bit_length() != 2048:
    q, ell_q = generate_sophie_germain_prime(q_bits)
    if q <= 2 * p:
        continue
    if is_mersenne(p) or is_mersenne(q):
        continue
    N = p * q

phi_N = (p - 1) * (q - 1)
e = 65537
d = mod_inverse(e, phi_N)

if d <= isqrt(N) // 12:
    raise ValueError("The primes are too small; we need to regenerate larger prime numbers.")


In [63]:
print(f"p = {p}")
print(f"q = {q}")
print(f"N = {N}")
print(f"e = {e}")
print(f"d = {d}")

p = 25557077787969209234521748572442186164752434229458229757457172159769297898396200525651175538115535271127149622647726605601682744151637943644428872206900322455598199019513474720526849634483546189249641881456341997197688919619218811982791766866719339147486592644680894218832796543676297995320947960718599325719
q = 694915494915072643042965470955162366341539881141320574760471356120872336344515683407852211056864294717047139788803904159253245170908478119895546618944033408916370841652741645126007782261714777553218342868753838156271185134712396749526594493020590432048318256509362095157863009779798676445948869455511356925499
N = 177600093596096330118886925432140962093060961689986562124264920897529806334345546306281088589680864016212273205065738971075587389085769791638324863406138158069809516477685341356496190816905224697709769120770551961149807523728966309233957061855197246769469183187701857308860553312091752654604301135213970768876023106632404193011022931991247479947618478006459280246693256

From now let's take

In [70]:
p = 38527196240183452608079249782597305012224194871958572512362208784794259775833758214872400372178703095612298723699476664828271804328598105003808673085755417960165135278228122186923094546829845970660073243005915727792146175361351288652136571391712332732430131233755841075907589274655499367647481193461807306083
q = 644044470721642297632118157751821630191322565564761131587686017233297352389191491301692037954483543419089938069767642458444196391630596064107032156058231534781117307819163613436857015901826987653197896363793547563740886912318567451940470703486246437748382500376874875507529729154692690929569810312885941144983
N = 24813227710897798853830008896403119989688160472559226789320675030328183383001690514237442707003901780891956548512711477002218066694353209195997040246304479578181546792001451803807680457615326551245466689489054236457918877601929505401920909987795408039254085172647603184441614178789602595230701733506069572915948120649660986611820287541101926006067604573307561627865995039545003472192917548495145467326712288268493463874602062600476729810452758851509852796820561459508329672028763432620764101481444561988023398481027708364320545162931609355624131490547779794554066924044804662135052210163467357921546626127687760831589
e = 65537
d = 4467266944792760252625849138176303656840114826979054837529863202203979976746524045615264453666463785537088901168828031755331659504198750557754689379528305454063568222512857314694398915412717060258255053928030745013762986356183014697609973250926927071046263194105147778708616593611830279401361822384883575550104799005857262294669934288813388100296397731572440970116337434621725636456796942729271671897883492418831845954656319301883410635052101403598363400927962869910804434962864559758112426723772256999386421338892824871657534750994014031261073198661644845858680247744492704633466654066723714219440343915607836887221

These keys are all secreted by sophie germain prime algorithm

So, here we have our robust keys.

Now let's test the practical application of this.

Suppose Alice wants to send Bob a message.  
`message = "Hello Bob, the code to your door is 1034"`  
This message must remain secret.

In [71]:
message = "Hello Bob, the code to your door is 1034"

Alice will convert it into an integer.

In [72]:
message_bytes = message.encode('ascii')
message_hex = message_bytes.hex()
M = int(message_hex, 16)
print("Message original :", message)
print("Message en hex   :", message_hex)
print("Message en entier:", M)

Message original : Hello Bob, the code to your door is 1034
Message en hex   : 48656c6c6f20426f622c2074686520636f646520746f20796f757220646f6f722069732031303334
Message en entier: 604052007910143330196099754351037768595400200209858406957808296347229419006803528940375679972148


And after that, Alice will encrypt it.

In [73]:
C = pow(M, e, N)
print("\n Encrypted message (C) :", C)


 Encrypted message (C) : 14675713410737394798596162373577250436861761201799842016502027372693731926344725602648865280962875806953022629792180268891987143997093506066708000618504605525761636739445191138271510679787675655006738263361638199818454553371076625783927247447885568491387454244508549288834554245197516127154581139874885573231813378390906985736634731392688189040221665841592294085676176169110537608584771262315654983024559730769272779733108883559469002849153620579407065844397478109964125312703127256360366052142824889668189567021970611024108693397900274620154972361988236616047231510168098563887629165470015249819156394558275531886277


So here is the encrypted message.  
Someone can see it, but they do not know its content.

Alice sends it to Bob.  
To decrypt it, Bob does the following:

In [74]:
ciphertext = pow(C, d, N)
print("\nDecrypted message (integer) :", ciphertext)


Decrypted message (integer) : 604052007910143330196099754351037768595400200209858406957808296347229419006803528940375679972148


Here is the fully decrypted message.

By converting it back from the integer, we obtain this:

In [75]:
hex_decrypt = hex(ciphertext)[2:]  
if len(hex_decrypt) % 2 != 0:
    hex_decrypt = "0" + hex_decrypt
final_message = bytes.fromhex(hex_decrypt).decode('ascii')


print("the message  :", final_message)


the message  : Hello Bob, the code to your door is 1034
